# 🌍 AirSentinel Cameroun
## Notebook 04 — Modélisation
**Responsables : FANKAM Marc Aurel + PEURBAR Firmin — ISSEA**

### Objectif
Prédire PM2.5 depuis les variables météo
Conformément à l'objectif du hackathon IndabaX 2026

### Split temporel
- Train : juillet 2022 → décembre 2024
- Test  : janvier 2025 → décembre 2025

### Modèles comparés
1. Régression Linéaire (baseline)
2. Random Forest
3. XGBoost
4. LightGBM

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import lightgbm as lgb
import joblib, os, warnings
warnings.filterwarnings('ignore')

os.makedirs('../models', exist_ok=True)

df = pd.read_excel('../data/processed/dataset_enrichi.xlsx')
df['date'] = pd.to_datetime(df['date'])
VAR_CIBLE = 'pm2_5_moyen'
print(f'✅ Dataset chargé : {df.shape[0]:,} lignes')

✅ Dataset chargé : 44,415 lignes


## Étape 1 — Définir les features météo

In [2]:
# Features météo uniquement (disponibles 2020-2025)
# Conformément à l'objectif du hackathon :
# 'Prédire les indicateurs de pollution depuis les données météorologiques'

FEATURES_METEO = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'apparent_temperature_max', 'apparent_temperature_min',
    'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant',
    'precipitation_sum', 'rain_sum', 'precipitation_hours',
    'sunshine_duration', 'shortwave_radiation_sum',
    'daylight_duration', 'et0_fao_evapotranspiration',
    # Variables créées (feature engineering)
    'mois', 'jour_annee', 'saison_code',
    'pm2_5_lag_1j', 'pm2_5_lag_3j', 'pm2_5_lag_7j',
    'pm2_5_moy_7j', 'pm2_5_moy_30j',
    'indice_stagnation',
    'vague_chaleur', 'harmattan_intense', 'episode_feux', 'humidite_infectieuse',
    # Géographie
    'latitude', 'longitude'
]

# Garder seulement les features disponibles
features = [f for f in FEATURES_METEO if f in df.columns]
print(f'Features disponibles : {len(features)}')
print(features)

Features disponibles : 25
['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'apparent_temperature_max', 'apparent_temperature_min', 'wind_speed_10m_max', 'wind_gusts_10m_max', 'wind_direction_10m_dominant', 'precipitation_sum', 'rain_sum', 'precipitation_hours', 'sunshine_duration', 'shortwave_radiation_sum', 'daylight_duration', 'et0_fao_evapotranspiration', 'mois', 'jour_annee', 'saison_code', 'pm2_5_lag_1j', 'pm2_5_lag_3j', 'pm2_5_lag_7j', 'pm2_5_moy_7j', 'pm2_5_moy_30j', 'latitude', 'longitude']


## Étape 2 — Split temporel
**Règle absolue : jamais de split aléatoire = fuite de données**

In [3]:
# Garder seulement les lignes avec PM2.5 connu
df_model = df[df[VAR_CIBLE].notna()].copy()

# Split temporel strict
train = df_model[df_model['date'] < '2025-01-01']
test  = df_model[df_model['date'] >= '2025-01-01']

X_train = train[features].fillna(train[features].median())
y_train = train[VAR_CIBLE]
X_test  = test[features].fillna(train[features].median())
y_test  = test[VAR_CIBLE]

print(f'Train : {len(train):,} lignes ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'Test  : {len(test):,} lignes  ({test["date"].min().date()} → {test["date"].max().date()})')

TypeError: Cannot convert [[datetime.datetime(2026, 9, 22, 0, 0)
  datetime.datetime(2026, 2, 28, 0, 0) '29.0' ...
  datetime.datetime(2026, 5, 31, 0, 0) '32.5' '32.7']
 [datetime.datetime(2026, 1, 19, 0, 0)
  datetime.datetime(2026, 3, 19, 0, 0)
  datetime.datetime(2026, 8, 18, 0, 0) ...
  datetime.datetime(2026, 4, 18, 0, 0)
  datetime.datetime(2026, 2, 17, 0, 0)
  datetime.datetime(2026, 8, 16, 0, 0)]
 [datetime.datetime(2026, 5, 20, 0, 0)
  datetime.datetime(2026, 1, 23, 0, 0)
  datetime.datetime(2026, 7, 22, 0, 0) ...
  datetime.datetime(2026, 9, 23, 0, 0)
  datetime.datetime(2026, 2, 24, 0, 0)
  datetime.datetime(2026, 9, 23, 0, 0)]
 ...
 ['1.84' '3.91' '3.57' ... '4.52' '4.83' '4.78']
 ['3.98' '3.98' '3.98' ... '3.51' '3.51' '3.51']
 ['13.17' '13.17' '13.17' ... datetime.datetime(2026, 5, 15, 0, 0)
  datetime.datetime(2026, 5, 15, 0, 0)
  datetime.datetime(2026, 5, 15, 0, 0)]] to numeric

## Étape 3 — Entraîner les 4 modèles

In [4]:
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

modeles = {
    'Régression Linéaire': (LinearRegression(), True),
    'Random Forest':       (RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), False),
    'XGBoost':             (xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbosity=0), False),
    'LightGBM':            (lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1), False),
}

resultats, entraines = [], {}
for nom, (mod, scaled) in modeles.items():
    Xtr = X_train_s if scaled else X_train
    Xte = X_test_s  if scaled else X_test
    mod.fit(Xtr, y_train)
    pred = mod.predict(Xte)
    r2   = r2_score(y_test, pred)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    resultats.append({'Modèle': nom, 'R²': round(r2,4), 'MAE': round(mae,3), 'RMSE': round(rmse,3)})
    entraines[nom] = mod
    print(f'{nom:30s} → R²={r2:.3f}  MAE={mae:.2f}  RMSE={rmse:.2f}')

df_res   = pd.DataFrame(resultats).sort_values('R²', ascending=False)
meilleur = df_res.iloc[0]['Modèle']
print(f'\n✅ Meilleur modèle : {meilleur}')
print(df_res.to_string(index=False))

NameError: name 'X_train' is not defined

## Étape 4 — Graphique prédiction vs réel

In [ ]:
pred_final = entraines[meilleur].predict(
    X_test_s if meilleur == 'Régression Linéaire' else X_test
)

plt.figure(figsize=(10, 5))
plt.scatter(y_test, pred_final, alpha=0.3, s=5)
plt.plot([0, y_test.max()], [0, y_test.max()], 'r--', label='Prédiction parfaite')
plt.xlabel('PM2.5 réel (µg/m³)')
plt.ylabel('PM2.5 prédit (µg/m³)')
plt.title(f'Prédiction vs Réel — {meilleur} — R²={df_res.iloc[0]["R²"]}')
plt.legend()
plt.tight_layout()
plt.savefig('../graphiques/prediction_vs_reel.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Graphique sauvegardé')

## Étape 5 — Sauvegarder les modèles

In [ ]:
joblib.dump(entraines[meilleur], '../models/meilleur_modele.pkl')
joblib.dump(scaler,              '../models/scaler.pkl')
joblib.dump(features,            '../models/features.pkl')
df_res.to_excel('../data/processed/comparaison_modeles.xlsx', index=False)

print('✅ Modèles sauvegardés :')
print('   models/meilleur_modele.pkl')
print('   models/scaler.pkl')
print('   models/features.pkl')